## clone the GitHub repo

In [1]:
!git clone https://github.com/siltanukifilie/aims-ktt-ai-math-tutor.git
%cd aims-ktt-ai-math-tutor

Cloning into 'aims-ktt-ai-math-tutor'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 62 (delta 18), reused 51 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 33.40 KiB | 3.34 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/aims-ktt-ai-math-tutor


## install required package

In [2]:
!pip install scikit-learn

## create the evaluation code

In [6]:
%%writefile kt_eval.py
import random
from sklearn.metrics import roc_auc_score

from tutor.curriculum_loader import load_curriculum


# Updated skills (FIXED)
SKILLS = [
    "counting",
    "number_sense",
    "addition",
    "subtraction",
    "multiplication",
    "division",
    "word_problem",
]


def generate_replay(num_events=300):
    """
    Generate synthetic learner replay.
    """
    curriculum = load_curriculum()
    replay = []

    learner_mastery = {
        skill: random.uniform(0.25, 0.75)
        for skill in SKILLS
    }

    for _ in range(num_events):
        item = random.choice(curriculum)
        skill = item.get("skill", "counting")
        difficulty = item.get("difficulty", 1)

        # Ensure skill exists
        if skill not in learner_mastery:
            learner_mastery[skill] = 0.4

        mastery = learner_mastery[skill]

        difficulty_penalty = difficulty / 12
        prob_correct = max(0.05, min(0.95, mastery - difficulty_penalty + 0.35))

        correct = random.random() < prob_correct

        # Update mastery
        if correct:
            learner_mastery[skill] = min(0.95, mastery + 0.05)
        else:
            learner_mastery[skill] = max(0.05, mastery - 0.03)

        replay.append(
            {
                "skill": skill,
                "difficulty": difficulty,
                "correct": int(correct),
            }
        )

    return replay


def bkt_predict_and_update(replay):
    """
    BKT-style model
    """
    mastery = {skill: 0.30 for skill in SKILLS}

    predictions = []
    labels = []

    for event in replay:
        skill = event["skill"]
        correct = event["correct"]

        # SAFETY FIX
        if skill not in mastery:
            mastery[skill] = 0.30

        pred = mastery[skill]
        predictions.append(pred)
        labels.append(correct)

        if correct:
            mastery[skill] = mastery[skill] + 0.15 * (1 - mastery[skill])
        else:
            mastery[skill] = mastery[skill] - 0.10 * mastery[skill]

        mastery[skill] = max(0.05, min(0.95, mastery[skill]))

    return labels, predictions


def elo_predict_and_update(replay):
    """
    Elo-style baseline
    """
    skill_rating = {skill: 1000 for skill in SKILLS}

    predictions = []
    labels = []

    for event in replay:
        skill = event["skill"]
        difficulty = event["difficulty"]
        correct = event["correct"]

        # SAFETY FIX
        if skill not in skill_rating:
            skill_rating[skill] = 1000

        item_rating = 900 + difficulty * 50

        expected = 1 / (1 + 10 ** ((item_rating - skill_rating[skill]) / 400))

        predictions.append(expected)
        labels.append(correct)

        k = 32
        skill_rating[skill] = skill_rating[skill] + k * (correct - expected)

    return labels, predictions


def run_eval():
    random.seed(42)

    replay = generate_replay(num_events=300)

    split = int(len(replay) * 0.7)
    held_out = replay[split:]

    bkt_labels, bkt_preds = bkt_predict_and_update(held_out)
    elo_labels, elo_preds = elo_predict_and_update(held_out)

    bkt_auc = roc_auc_score(bkt_labels, bkt_preds)
    elo_auc = roc_auc_score(elo_labels, elo_preds)

    print("Knowledge Tracing Evaluation")
    print("----------------------------")
    print(f"Replay events: {len(replay)}")
    print(f"Held-out events: {len(held_out)}")
    print(f"BKT AUC: {bkt_auc:.3f}")
    print(f"Elo baseline AUC: {elo_auc:.3f}")

    if bkt_auc >= elo_auc:
        print("Interpretation: BKT performs as well as or better than the Elo baseline.")
    else:
        print("Interpretation: Elo performs better; more real data would improve BKT.")


if __name__ == "__main__":
    run_eval()

Overwriting kt_eval.py


In [7]:
!python kt_eval.py

Knowledge Tracing Evaluation
----------------------------
Replay events: 300
Held-out events: 90
BKT AUC: 0.913
Elo baseline AUC: 0.883
Interpretation: BKT performs as well as or better than the Elo baseline.
